In [2]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554979 sha256=d636a08058dc9e99e9f0e13d92bfd33d95410df3327431de10548d81655acb5b
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [6]:
 !pip install "numpy<2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4

In [1]:
import polars as pl
import pandas as pd
from surprise import SVD, Dataset, Reader, accuracy
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'

In [4]:
split_dir = f"{drive_path}/processed_data/final_foundation/splits"

In [5]:
train_pl = pl.read_parquet(f"{split_dir}/train.parquet")
test_pl = pl.read_parquet(f"{split_dir}/test.parquet")

In [6]:
# 3. Memory Management: Sample 2 million rows for training
# This ensures the model fits in the Colab T4 RAM limits
train_sample = train_pl.sample(n=2_000_000, seed=42).to_pandas()
test_pd = test_pl.to_pandas()

In [7]:
# 4. Prepare the data for Surprise
# BGG ratings are on a scale of 1 to 10
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(train_sample[['user_id', 'item_id', 'Rating']], reader)
trainset = data.build_full_trainset()

In [8]:
print("Training SVD model (this may take 2-5 minutes)...")
algo = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo.fit(trainset)

Training SVD model (this may take 2-5 minutes)...


In [9]:
# 6. Evaluate on the Test Set
print("Evaluating on test set...")
# Convert test set to Surprise format: list of (uid, iid, r_ui) tuples
test_tuples = list(test_pd[['user_id', 'item_id', 'Rating']].itertuples(index=False, name=None))
predictions = algo.test(test_tuples)

Evaluating on test set...


In [10]:
# Calculate RMSE
rmse_svd = accuracy.rmse(predictions)
print(f"SVD Baseline RMSE: {rmse_svd:.4f}")

RMSE: 1.3064
SVD Baseline RMSE: 1.3064
